## Data Reading

In [0]:
# Datafame reader API spark.read
# use option inferSchema to auto fetch schema of file will see first 2-3 records and infer data types from it
# Add header option to fetch first record as header
# DataFrame reader API usually use to load data from all files doc_link: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.html
df=spark.read.format("csv").option("inferSchema",True).option("header",True).load('/Volumes/poc/default/myvolume/BigMart Sales.csv')

In [0]:
# df.display()
display(df.limit(10))

Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
FDA15,9.3,Low Fat,0.016047301,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.138
DRC01,5.92,Regular,0.019278216,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
FDN15,17.5,Low Fat,0.016760075,Meat,141.618,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.27
FDX07,19.2,Regular,0.0,Fruits and Vegetables,182.095,OUT010,1998,null,Tier 3,Grocery Store,732.38
NCD19,8.93,Low Fat,0.0,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052
FDP36,10.395,Regular,0.0,Baking Goods,51.4008,OUT018,2009,Medium,Tier 3,Supermarket Type2,556.6088
FDO10,13.65,Regular,0.012741089,Snack Foods,57.6588,OUT013,1987,High,Tier 3,Supermarket Type1,343.5528
FDP10,null,Low Fat,0.127469857,Snack Foods,107.7622,OUT027,1985,Medium,Tier 3,Supermarket Type3,4022.7636
FDH17,16.2,Regular,0.016687114,Frozen Foods,96.9726,OUT045,2002,null,Tier 2,Supermarket Type1,1076.5986
FDU28,19.2,Regular,0.09444959,Frozen Foods,187.8214,OUT017,2007,null,Tier 2,Supermarket Type1,4710.535


In [0]:
df_json=spark.read.format("json")\
                .option("inferSchema",True)\
                .option("header",True)\
                .option("multiLine",False)\
                .load('/Volumes/poc/default/myvolume/drivers.json')

df_json.display()

## Schema-DDL and StructType()

In [0]:
# if we are not mentioning inferSchema option then we have to manually specify schema 
# there are 2 ways to define schema 1. using struct type 2. using DDL

In [0]:
df.printSchema()

root
 |-- Item_Identifier: string (nullable = true)
 |-- Item_Weight: double (nullable = true)
 |-- Item_Fat_Content: string (nullable = true)
 |-- Item_Visibility: double (nullable = true)
 |-- Item_Type: string (nullable = true)
 |-- Item_MRP: double (nullable = true)
 |-- Outlet_Identifier: string (nullable = true)
 |-- Outlet_Establishment_Year: integer (nullable = true)
 |-- Outlet_Size: string (nullable = true)
 |-- Outlet_Location_Type: string (nullable = true)
 |-- Outlet_Type: string (nullable = true)
 |-- Item_Outlet_Sales: double (nullable = true)



In [0]:
# one way to define schema 

ddl_schema = """
Item_Identifier STRING,
Item_Weight STRING,
Item_Fat_Content STRING,
Item_Visibility DOUBLE,
Item_Type STRING,
Item_MRP DOUBLE,
Outlet_Identifier STRING,
Outlet_Establishment_Year INT,
Outlet_Size STRING,
Outlet_Location_Type STRING,
Outlet_Type STRING,
Item_Outlet_Sales DOUBLE
"""

df=spark.read.format("csv").schema(ddl_schema).option("header",True).load('/Volumes/poc/default/myvolume/BigMart Sales.csv')
df.display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Second way to define schema 

schema = StructType([
    StructField("Item_Identifier", StringType(), True),
    StructField("Item_Weight", StringType(), True),
    StructField("Item_Fat_Content", StringType(), True),
    StructField("Item_Visibility", DoubleType(), True),
    StructField("Item_Type", StringType(), True),
    StructField("Item_MRP", DoubleType(), True),
    StructField("Outlet_Identifier", StringType(), True),
    StructField("Outlet_Establishment_Year", IntegerType(), True),
    StructField("Outlet_Size", StringType(), True),
    StructField("Outlet_Location_Type", StringType(), True),
    StructField("Outlet_Type", StringType(), True),
    StructField("Item_Outlet_Sales", DoubleType(), True)
])
df=spark.read.format("csv").schema(schema).option("header",True).load('/Volumes/poc/default/myvolume/BigMart Sales.csv')
df.printSchema()

root
 |-- Item_Identifier: string (nullable = true)
 |-- Item_Weight: string (nullable = true)
 |-- Item_Fat_Content: string (nullable = true)
 |-- Item_Visibility: double (nullable = true)
 |-- Item_Type: string (nullable = true)
 |-- Item_MRP: double (nullable = true)
 |-- Outlet_Identifier: string (nullable = true)
 |-- Outlet_Establishment_Year: integer (nullable = true)
 |-- Outlet_Size: string (nullable = true)
 |-- Outlet_Location_Type: string (nullable = true)
 |-- Outlet_Type: string (nullable = true)
 |-- Item_Outlet_Sales: double (nullable = true)



## SELECT

In [0]:
# SELECT statement, calling the columns

# 1 way
df.select("Item_Identifier","Item_Weight","Item_Fat_Content").display()

In [0]:
# 2nd way preferable
# with col() it defines that this is the column object
df.select(col("Item_Identifier"),col("Item_Weight"),col("Item_Fat_Content")).display()

## Alias

In [0]:
df.select(col('Item_Identifier').alias('Item_ID')).display()

## Filter/Where

In [0]:
df.filter(col('Item_Fat_Content')=='Regular').display()

In [0]:
df.filter((col("Item_Type")=='Soft Drinks') & (col("Item_Weight")<10)).display()

In [0]:
df.filter((col('Outlet_Size').isNull() & col("Outlet_Location_Type").isin('Tier 3','Tier 2'))).display()

## Column Renamed 

In [0]:
df.withColumnRenamed('Item_Weight','Item_Wt').display()

## WithColumn

In [0]:
# to add new column, to modify existing column 

df=df.withColumn('flag',lit("new"))
df.display()

In [0]:
df.withColumn('multiply',col('Item_Weight')*col('Item_MRP')).display()

In [0]:
df.withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),'Low Fat','LF')).display()

## Type Casting

In [0]:
df.withColumn('Item_weight',col("Item_Weight").cast(StringType())).display()

## Sort/OrderBy

In [0]:
df.sort(col('Item_Weight').desc()).display()

In [0]:
df.sort(col('Item_Visibility').asc()).display()

In [0]:
# sorting base on multiple columns pick both column for descending [0,0]
df.sort(['Item_Visibility','Item_Weight'],ascending=[0,0]).display()

In [0]:
# sorting base on multiple columns pick one column as descending and one as ascending[0,0]
df.sort(['Item_Visibility','Item_Weight'],ascending=[0,1]).display()

## Limit

In [0]:
df.limit(10).display()

Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,flag
FDA15,9.3,Low Fat,0.016047301,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.138,new
DRC01,5.92,Regular,0.019278216,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228,new
FDN15,17.5,Low Fat,0.016760075,Meat,141.618,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.27,new
FDX07,19.2,Regular,0.0,Fruits and Vegetables,182.095,OUT010,1998,null,Tier 3,Grocery Store,732.38,new
NCD19,8.93,Low Fat,0.0,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052,new
FDP36,10.395,Regular,0.0,Baking Goods,51.4008,OUT018,2009,Medium,Tier 3,Supermarket Type2,556.6088,new
FDO10,13.65,Regular,0.012741089,Snack Foods,57.6588,OUT013,1987,High,Tier 3,Supermarket Type1,343.5528,new
FDP10,null,Low Fat,0.127469857,Snack Foods,107.7622,OUT027,1985,Medium,Tier 3,Supermarket Type3,4022.7636,new
FDH17,16.2,Regular,0.016687114,Frozen Foods,96.9726,OUT045,2002,null,Tier 2,Supermarket Type1,1076.5986,new
FDU28,19.2,Regular,0.09444959,Frozen Foods,187.8214,OUT017,2007,null,Tier 2,Supermarket Type1,4710.535,new


## Drop

In [0]:
df.drop("Item_Fat_Content").display()

In [0]:
df.drop("Item_Fat_Content",'Item_Type').display()

## Drop Duplicates

In [0]:
df.dropDuplicates().display()

In [0]:
# drop duplicatees bbases on column
df.dropDuplicates(subset=["Item_Type"]).display()

In [0]:
df.distinct().display()

## UNION and UnionByName

In [0]:
data1 = [('1','kad'),
        ('2','sid')]
schema1 = 'id STRING, name STRING' 

df1 = spark.createDataFrame(data1,schema1)

data2 = [('3','rahul'),
        ('4','jas')]
schema2 = 'id STRING, name STRING' 

df2 = spark.createDataFrame(data2,schema2)

In [0]:
df1.union(df2).display()

id,name
1,kad
2,sid
3,rahul
4,jas


In [0]:
# simple UNION will not take care of this case, to catter this we have UnionByName
data1 = [('kad','1',),
        ('sid','2',)]
schema1 = 'name STRING, id STRING' 

df1 = spark.createDataFrame(data1,schema1)

df1.union(df2).display()

name,id
kad,1
sid,2
3,rahul
4,jas


In [0]:
df1.unionByName(df2).display()

name,id
kad,1
sid,2
rahul,3
jas,4


## String Functions

In [0]:
df.select(lower('Item_type').alias('Item_type')).display()
df.select(upper('Item_type'))

## Date Functions

In [0]:
# Current Date
df=df.withColumn('current_date',current_date())

In [0]:
# Date Add
df=df.withColumn('week_after', date_add('current_date',7))

In [0]:
# Date Sub
df.withColumn('week_before', date_sub('current_date',7)).display()

In [0]:
 df.withColumn('week_before', date_add('current_date',-7)).display()

In [0]:
# Date Difference

df.withColumn('datediff',datediff('current_date','week_after')).display()

In [0]:
# DateFormat
df.withColumn('week_after',date_format('week_after','dd-MM-yyyy')).display()

## Handling Nulls

In [0]:
# it will drop all the records which have NULL in all the columns
df.dropna('all')

DataFrame[Item_Identifier: string, Item_Weight: double, Item_Fat_Content: string, Item_Visibility: double, Item_Type: string, Item_MRP: double, Outlet_Identifier: string, Outlet_Establishment_Year: int, Outlet_Size: string, Outlet_Location_Type: string, Outlet_Type: string, Item_Outlet_Sales: double, flag: string, current_date: date, week_after: date]

In [0]:
# it will drop all the records which have NULL in any columns
df.dropna('any').display()

In [0]:
# will drop all the column which have NULL in only specific column
df.dropna('any',subset=['Outlet_Size']).display()

In [0]:
# fill null values
df=df.fillna('Not Available yet') # will fill all the NULL values in all columns with 'Not Available yet'
df=df.fillna(1000000,subset=["Item_Weight"]) # will fill all the NULL values in Item_Weight column with 1000000
df.display()

## Split and Indexing

In [0]:
from pyspark.sql.functions import *
df.withColumn('Outlet_Type',split('Outlet_Type',' '))
df.withColumn('Outlet_Type',split('Outlet_Type',' ')[0]).display() # split with Indexing

## EXPLODE

In [0]:
# explode woorks with array
df_explode=df.withColumn('Outlet_Type',split('Outlet_Type',' '))
df_explode.withColumn('Outlet_Type',explode('Outlet_Type')).display()

In [0]:
df_explode.withColumn('Type1_Flag',array_contains('Outlet_Type','Type1')).display()

In [0]:
# GroupBy
df.groupBy('Item_Type').agg(sum("Item_MRP").alias("Total_MRP"))
df.groupBy(['Item_Type','Outlet_Size']).agg(avg("Item_MRP").alias("Avg_MRP")).display()

In [0]:
df.groupBy(['Item_Type','Outlet_Size']).agg(avg("Item_MRP").alias("Avg_MRP"),sum("Item_MRP").alias("Total_MRP")).display()

Item_Type,Outlet_Size,Avg_MRP,Total_MRP
Fruits and Vegetables,Medium,142.9714702179177,59047.217200000014
Health and Hygiene,Medium,128.70186470588237,21879.317000000003
Meat,High,137.2447902439025,5627.036400000002
Meat,Small,145.69925042016808,17338.2108
Seafood,Not Available yet,142.21686666666668,2559.9036
Breakfast,Small,130.56802666666667,3917.0407999999998
Soft Drinks,Not Available yet,133.42344360902257,17745.318000000003
Breakfast,Not Available yet,158.6750903225807,4918.927800000001
Seafood,Medium,140.857619047619,2958.0099999999993
Breads,Not Available yet,139.04861666666667,10011.5004


In [0]:
# Collect list

data = [('user1','book1'),
        ('user1','book2'),
        ('user2','book2'),
        ('user2','book4'),
        ('user3','book1')]

schema = 'user string, book string'

df_book = spark.createDataFrame(data,schema)

df_book.groupBy('user').agg(collect_list('book').alias('books')).display()

user,books
user1,"List(book1, book2)"
user2,"List(book2, book4)"
user3,List(book1)


In [0]:
# Pivoting
df.groupBy('Item_Type').pivot('Outlet_Size').agg(avg("Item_MRP")).display()

Item_Type,High,Medium,Not Available yet,Small
Fruits and Vegetables,145.57287042253515,142.9714702179177,142.57516045845267,148.31336951219507
Health and Hygiene,135.11098032786884,128.70186470588237,130.55989019607844,131.83153529411757
Meat,137.2447902439025,136.41913154362408,139.29453448275865,145.69925042016808
Seafood,134.86424000000002,140.857619047619,142.21686666666668,144.28176
Breakfast,147.49058461538462,134.53751111111112,158.6750903225807,130.56802666666667
Soft Drinks,131.75847346938772,128.2696817518248,133.42344360902257,132.8550428571429
Breads,133.75896,140.8610385542169,139.04861666666667,145.5236507042254
Baking Goods,129.20204383561642,126.17856847290639,126.66939891891889,125.21336363636368
Others,132.5766125,127.83618076923078,132.59299565217393,137.88921090909088
Frozen Foods,136.82925,140.55701532846714,137.49448464730293,137.83854377510033


## Joins

In [0]:
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'inner').display()

emp_id,emp_name,dept_id,dept_id,department
1,gaur,d01,d01,HR
2,kit,d02,d02,Marketing
3,sam,d03,d03,Accounts
4,tim,d03,d03,Accounts
5,aman,d05,d05,Finance


In [0]:
# left join
df1.join(df2,df1['dept_id']==df2['dept_id'],'left').display()

emp_id,emp_name,dept_id,dept_id,department
1,gaur,d01,d01,HR
2,kit,d02,d02,Marketing
3,sam,d03,d03,Accounts
4,tim,d03,d03,Accounts
5,aman,d05,d05,Finance
6,nad,d06,null,null


In [0]:
# Right join
df1.join(df2,df1['dept_id']==df2['dept_id'],'right').display()

emp_id,emp_name,dept_id,dept_id,department
1,gaur,d01,d01,HR
2,kit,d02,d02,Marketing
4,tim,d03,d03,Accounts
3,sam,d03,d03,Accounts
null,null,null,d04,IT
5,aman,d05,d05,Finance


In [0]:
# Anti Join: will fetch those records which are not matched
df1.join(df2,df1['dept_id']==df2['dept_id'],'anti').display()

emp_id,emp_name,dept_id
6,nad,d06


## Window Functions

In [0]:
from pyspark.sql.window import Window

df.withColumn('rowCol',row_number().over(Window.orderBy('Item_Identifier'))).display()

In [0]:
df.withColumn('rank',rank().over(Window.orderBy(col('Item_Identifier').desc())))\
            .withColumn('dense_rank',dense_rank().over(Window.orderBy(col('Item_Identifier').desc()))).display()

In [0]:
# cumulative sum
df.withColumn('cum_sum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()

In [0]:
df.withColumn('total_sum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))).display()

## User Defined Functions

In [0]:
# if we want to perform transformation that can not be acheived by using predefine functions then we can use udf
# we should'nt use userdefine functions, try to avoid as much as wee can , prefer to use spark libraries

# step1
def funct(x):
    return x*x

# step2
my_udf=udf(funct)

# step3
df.withColumn('mynewcol',my_udf('Item_MRP')).display()


## Data Writting

In [0]:
df.write.format('csv')\
    .save('/Volumes/poc/default/myvolume/sample.csv')

In [0]:
# Writting Modes 

# Append: use when we dont want to loose our data , we have file already there and we want to append new data to it , it is foler level if we have file in folder we just append new file in that folder

# Overwrite: delete prvious file in foler and place that file you want to add

# Error: will throw error if file already exists

# Ignore: if found file in folder it ignore it no operation perform

In [0]:
# again push data to same folder using append

df.write.format('csv')\
    .mode('append')\
    .save('/Volumes/poc/default/myvolume/sample.csv')

In [0]:
# again push data to same folder using overwrite

df.write.format('csv')\
    .mode('overwrite')\
    .save('/Volumes/poc/default/myvolume/sample.csv')

In [0]:
# again push data to same folder using Error 

# this code will through error
df.write.format('csv')\
    .mode('error')\
    .save('/Volumes/poc/default/myvolume/sample.csv')

In [0]:
# again push data to same folder using Ignore

df.write.format('csv')\
    .mode('ignore')\
    .save('/Volumes/poc/default/myvolume/sample.csv')

In [0]:
# parquet file format , in parquet file header/metadata store in footer and delta lake metadata store in transaction logs

# this code will overwrite previous csv files
df.write.format('parquet')\
    .mode('overwrite')\
    .save('/Volumes/poc/default/myvolume/sample.csv')

In [0]:
# make table from df
df.write.format('parquet')\
    .mode('overwrite')\
    .saveAsTable('sample')

## Spark SQL

In [0]:
df.createOrReplaceTempView('sample')

In [0]:
spark.sql("SELECT * FROM sample").display()